In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from itertools import cycle
import timm
from tqdm.auto import tqdm

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
CONFIG = {
    "seed": 42,
    "img_size": 224,
    "batch_size": 64,          
    "num_epochs": 10,
    "weight_decay": 1e-4,
    "num_workers": 4,          
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "data_dir": "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000", 
    "output_dir": "/kaggle/working/"
}

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(CONFIG["seed"])
print(f"⚡ Device: {CONFIG['device']} | GPUs Available: {torch.cuda.device_count()}")

# ==============================================================================
# 2. LG-ViT CUSTOM ARCHITECTURE (Multi-Class)
# ==============================================================================
class LaplacianEdgeExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        kernel = torch.tensor([[0,  1, 0],
                               [1, -4, 1],
                               [0,  1, 0]], dtype=torch.float32)
        kernel = kernel.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.laplacian = nn.Conv2d(3, 3, kernel_size=3, padding=1, groups=3, bias=False)
        self.laplacian.weight.data = kernel
        self.laplacian.weight.requires_grad = False
        self.proj = nn.Conv2d(3, 768, kernel_size=16, stride=16)

    def forward(self, x):
        edges = self.laplacian(x)
        edges = self.proj(edges)
        edges = edges.flatten(2).transpose(1, 2)
        return edges

class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=768, num_heads=8):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        
    def forward(self, vit_features, edge_features):
        attn_out, _ = self.cross_attn(query=self.norm1(vit_features),
                                      key=self.norm2(edge_features),
                                      value=self.norm2(edge_features))
        return vit_features + attn_out

class LGViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
        self.edge_extractor = LaplacianEdgeExtractor()
        self.cross_attention = CrossAttentionBlock(dim=768)
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, num_classes)
        )

    def forward(self, x):
        vit_out = self.vit.forward_features(x)
        vit_patches = vit_out[:, 1:, :] 
        edge_patches = self.edge_extractor(x)
        fused_patches = self.cross_attention(vit_patches, edge_patches)
        pooled_features = fused_patches.mean(dim=1)
        return self.head(pooled_features)

# ==============================================================================
# 3. CSV DATALOADER WITH DEDUPLICATION
# ==============================================================================
class HAM10000Dataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image = Image.open(self.file_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

def prepare_dataloaders(base_path, config):
    print(f"🔍 Locating metadata in: {base_path}")
    csv_files = glob.glob(os.path.join(base_path, "**", "HAM10000_metadata.csv"), recursive=True)
    if not csv_files:
        raise ValueError(f"🚨 Could not find HAM10000_metadata.csv")
    
    df = pd.read_csv(csv_files[0])
    
    class_names = sorted(df['dx'].unique().tolist())
    config["num_classes"] = len(class_names)
    config["class_names"] = class_names
    print(f"🏷️ Detected {len(class_names)} Classes: {class_names}")
    
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    img_id_to_label = dict(zip(df['image_id'], df['dx'].map(class_to_idx)))
    
    print("🔍 Scanning and deduplicating images...")
    all_image_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"):
        all_image_paths.extend(glob.glob(os.path.join(base_path, "**", ext), recursive=True))
        
    # STRICT DEDUPLICATION: Ensure exactly 10,015 images
    unique_paths = {}
    for path in all_image_paths:
        img_id = os.path.splitext(os.path.basename(path))[0]
        if img_id in img_id_to_label and img_id not in unique_paths:
            unique_paths[img_id] = path
            
    valid_paths = list(unique_paths.values())
    valid_labels = [img_id_to_label[img_id] for img_id in unique_paths.keys()]
            
    print(f"📁 Successfully loaded exactly {len(valid_paths)} unique images.")

    all_paths = np.array(valid_paths)
    all_labels = np.array(valid_labels)

    # 70/15/15 Split
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        all_paths, all_labels, test_size=0.30, random_state=config["seed"], stratify=all_labels
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.50, random_state=config["seed"], stratify=temp_labels
    )

    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / np.sqrt(class_counts)
    class_weights = class_weights / np.sum(class_weights) * len(class_names)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(config["device"])
    print(f"⚖️ Square-Root Class Weights Applied: {np.round(class_weights.cpu().numpy(), 3)}")

    train_transform = transforms.Compose([
        transforms.Resize((config["img_size"], config["img_size"])),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_test_transform = transforms.Compose([
        transforms.Resize((config["img_size"], config["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(HAM10000Dataset(train_paths, train_labels, train_transform), batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(HAM10000Dataset(val_paths, val_labels, val_test_transform), batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(HAM10000Dataset(test_paths, test_labels, val_test_transform), batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"], pin_memory=True, persistent_workers=True)

    print(f"📊 Partitions: {len(train_paths)} Train | {len(val_paths)} Val | {len(test_paths)} Blind Test")
    return train_loader, val_loader, test_loader, class_weights

train_loader, val_loader, test_loader, class_weights = prepare_dataloaders(CONFIG["data_dir"], CONFIG)

# Instantiate LG-ViT Multi-class Model
lg_vit_model = LGViT(num_classes=CONFIG["num_classes"])
if torch.cuda.device_count() > 1:
    lg_vit_model = nn.DataParallel(lg_vit_model)
lg_vit_model = lg_vit_model.to(CONFIG["device"])

# ==============================================================================
# 4. DIFFERENTIAL FINE-TUNING LOOP (Weighted Loss)
# ==============================================================================
pretrained_backbone_params = []
custom_module_params = []

for name, param in lg_vit_model.named_parameters():
    if "vit" in name or "patch_embed" in name or "blocks" in name: 
        pretrained_backbone_params.append(param)
    else:
        custom_module_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': pretrained_backbone_params, 'lr': 1e-5},
    {'params': custom_module_params, 'lr': 3e-4}
], weight_decay=CONFIG["weight_decay"])

criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["num_epochs"])
scaler = torch.amp.GradScaler('cuda')

best_val_acc = 0.0
best_model_path = os.path.join(CONFIG["output_dir"], "best_lgvit_ham10000.pth")

print("\n🚀 Starting LG-ViT Differential Fine-Tuning on HAM10000...")

for epoch in range(1, CONFIG["num_epochs"] + 1):
    lg_vit_model.train()
    total_loss, correct, total = 0.0, 0, 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CONFIG['num_epochs']} [Train]")
    for images, labels in train_bar:
        images, labels = images.to(CONFIG["device"], non_blocking=True), labels.to(CONFIG["device"], non_blocking=True)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = lg_vit_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        train_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{correct/total:.4f}"})

    scheduler.step()
    
    lg_vit_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(CONFIG["device"], non_blocking=True), labels.to(CONFIG["device"], non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = lg_vit_model(images)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"📈 Epoch {epoch:02d} | Train Acc: {correct/total:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        raw_model = lg_vit_model.module if hasattr(lg_vit_model, "module") else lg_vit_model
        torch.save(raw_model.state_dict(), best_model_path)
        print(f"  ⭐ Best Model Saved (Val Acc: {best_val_acc:.4f})")

# ==============================================================================
# 5. MULTI-CLASS EVALUATION & VISUALIZATION
# ==============================================================================
print("\n🔬 Evaluating LG-ViT on HAM10000 Blind Test Set...")

if hasattr(lg_vit_model, "module"):
    lg_vit_model.module.load_state_dict(torch.load(best_model_path, map_location=CONFIG["device"]))
else:
    lg_vit_model.load_state_dict(torch.load(best_model_path, map_location=CONFIG["device"]))

lg_vit_model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating Test Set"):
        images = images.to(CONFIG["device"], non_blocking=True)
        with torch.amp.autocast('cuda'):
            outputs = lg_vit_model(images)
            probs = F.softmax(outputs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

print("\n" + "=" * 55)
print("📊 LG-ViT HAM10000 BLIND TEST SET CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(all_labels, all_preds, target_names=CONFIG["class_names"], digits=4))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".1%", cmap="Blues", xticklabels=CONFIG["class_names"], yticklabels=CONFIG["class_names"], cbar=False)
plt.title("Normalized Confusion Matrix (LG-ViT - HAM10000)", fontweight="bold")
plt.xlabel("Predicted Label", fontweight="bold")
plt.ylabel("True Label", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "lgvit_ham10000_cm.png"), dpi=300)
plt.show()

# Multi-Class ROC-AUC
labels_bin = label_binarize(all_labels, classes=range(CONFIG["num_classes"]))
macro_roc_auc = roc_auc_score(labels_bin, all_probs, average="macro", multi_class="ovr")

fpr, tpr, roc_auc_val = dict(), dict(), dict()
for i in range(CONFIG["num_classes"]):
    fpr[i], tpr[i], _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc_val[i] = auc(fpr[i], tpr[i])

colors = cycle(['blue', 'red', 'green', 'darkorange', 'purple', 'cyan', 'magenta'])
plt.figure(figsize=(9, 7))
for i, color in zip(range(CONFIG["num_classes"]), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2, label=f'{CONFIG["class_names"][i]} (AUC = {roc_auc_val[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label="Random Guess")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontweight="bold")
plt.ylabel("True Positive Rate", fontweight="bold")
plt.title(f"LG-ViT Multi-Class ROC-AUC (Macro AUC = {macro_roc_auc:.4f})", fontweight="bold")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "lgvit_ham10000_roc.png"), dpi=300)
plt.show()